# coherence-confidence — full experiment on 2x T4

One **Run all** executes the whole ladder: two models, five scored stages each,
analysis, and a zip of `results/` at the end.

**Before running — three settings:**

1. **Settings → Accelerator → GPU T4 x2** (both GPUs are required; the 8B in
   fp16 does not fit on one).
2. **Settings → Internet → ON** (clone + model downloads).
3. **Add-ons → Secrets**: add a secret named **`GITHUB_TOKEN`** and attach it
   to this notebook. To create the token: GitHub → Settings → Developer
   settings → Personal access tokens → **Fine-grained tokens** → Generate new.
   Resource owner: your account. Repository access: *Only select repositories*
   → `zacharyspeck/coherence-confidence`. Permissions → Repository permissions
   → **Contents: Read-only**. Nothing else. Copy the token into the Kaggle
   secret's value field.

Details, model choices, and expected runtime: `KAGGLE.md` in the repo.


In [ ]:
# --- 1. Clone the (private) repo using the GITHUB_TOKEN Kaggle secret -------
import os, pathlib, subprocess

from kaggle_secrets import UserSecretsClient

REPO = "zacharyspeck/coherence-confidence"
WORK = pathlib.Path("/kaggle/working")
tok = UserSecretsClient().get_secret("GITHUB_TOKEN")

os.chdir(WORK)
if not (WORK / "coherence-confidence").exists():
    # The token travels ONLY in process environment (GIT_CONFIG_* vars), never
    # in the URL - so it is never written to .git/config and there is no
    # window where an interrupted cell leaves it on disk. Kaggle preserves
    # /kaggle/working in saved notebook versions, which is why this matters.
    env = dict(
        os.environ,
        GIT_CONFIG_COUNT="1",
        GIT_CONFIG_KEY_0="http.extraheader",
        GIT_CONFIG_VALUE_0=f"Authorization: Bearer {tok}",
    )
    r = subprocess.run(
        ["git", "clone", "--depth", "1", f"https://github.com/{REPO}.git"],
        capture_output=True, text=True, env=env,
    )
    print((r.stdout + r.stderr).replace(tok, "***"))
    assert r.returncode == 0, "clone failed - is the GITHUB_TOKEN secret attached and valid?"

os.chdir(WORK / "coherence-confidence")

# The repo ships its own results/ history (CPU-era runs, audits). Move it
# aside so this session's results/ - and the zip in cell 5 - contain ONLY
# what THIS run produced. Idempotent across re-runs.
res, prior = pathlib.Path("results"), pathlib.Path("results_from_repo")
if res.exists() and not prior.exists():
    res.rename(prior)
res.mkdir(exist_ok=True)

print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)


In [ ]:
# --- 2. Install pinned requirements (minus the CPU torch) + GPU extras ------
# requirements.txt pins torch==2.13.0+cpu because the repo was built on a
# CPU-only laptop. Installing that here would REPLACE Kaggle's CUDA torch with
# the CPU wheel, so torch and its index line are filtered out; everything else
# keeps its pin (transformers==5.16.1 in particular).
import pathlib

req = pathlib.Path("requirements.txt").read_text().splitlines()
keep = [ln for ln in req
        if ln.strip()
        and not ln.startswith("#")
        and not ln.startswith("--extra-index-url")
        and not ln.startswith("torch==")]
pathlib.Path("/kaggle/working/req_gpu.txt").write_text("\n".join(keep))

%pip install -q -r /kaggle/working/req_gpu.txt bitsandbytes

import bitsandbytes, torch, transformers
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| gpus", torch.cuda.device_count())
print("transformers", transformers.__version__,
      "| bitsandbytes", bitsandbytes.__version__)
assert torch.cuda.is_available(), "no GPU - set Accelerator to GPU T4 x2"
assert torch.cuda.device_count() >= 2, "one GPU visible - set Accelerator to GPU T4 x2"


In [ ]:
# --- 3. Models + verify final-position logit reads under device_map=auto ----
# Ladder note (checked 2026-09): the Qwen3.6/3.8 27B models are MULTIMODAL
# (AutoModelForMultimodalLM, ~54GB bf16) and have no 8B sibling, so they fit
# neither src/score.py's causal-LM path nor this disk. The same-family
# text-only ladder is Qwen3-8B and Qwen3-32B; the 32B comes from unsloth's
# standard pre-quantized bnb-4bit mirror because the full bf16 checkpoint
# (~65GB) does not fit Kaggle's disk, and bnb quantize-at-load would still
# have to download all of it first.
#
# dtype: T4s (sm_75) have no bf16 units - float16 is the dtype on this
# hardware. That includes the 32B's 4-bit compute dtype: the unsloth repo
# bakes bnb_4bit_compute_dtype=bfloat16 into its config, and score.py
# overrides it to match --dtype at load (meta.bnb_compute_dtype_overridden
# records that it happened). --no-thinking is REQUIRED: Qwen3 hybrid models
# otherwise spend the next token on '<think>' and every option reads near
# zero.
MODELS = {
    "qwen3_8b":  "Qwen/Qwen3-8B",                 # fp16, sharded over both T4s
    "qwen3_32b": "unsloth/Qwen3-32B-bnb-4bit",    # pre-quantized nf4
}
COMMON = ["--device-map", "auto", "--dtype", "float16",
          "--chat-template", "--no-thinking", "--third-option", "Unknown"]

import gc, sys, torch
sys.path.insert(0, ".")
from src.models import load_items
from src.render import render_prompt
from src.score import HFScorer

OPTS = ("Yes", "No", "Unknown")
items = load_items(["items/draft", "items/seed"])[:2]

scorer = HFScorer(MODELS["qwen3_8b"], device_map="auto", dtype="float16",
                  chat_template=True, no_thinking=True, options=OPTS)

devs = (sorted({str(d) for d in scorer.model.hf_device_map.values()})
        if hasattr(scorer.model, "hf_device_map") else [scorer.device])
print("sharded over:", devs)
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} allocated {torch.cuda.memory_allocated(i)/1e9:.1f} GB")
assert len(devs) >= 2, "expected the 8B to shard across both T4s"

for it in items:
    prompt = render_prompt(it, OPTS)
    res = scorer.score_prompt(prompt)
    # Independent read of the SAME final position: a bare forward pass must
    # agree with the scorer's argmax, or device_map broke the logit indexing.
    enc = scorer.tokenizer(scorer._prepare(prompt), return_tensors="pt")
    enc = enc.to(scorer.model.device)
    with torch.no_grad():
        logits = scorer.model(**enc).logits[0, -1, :]
    manual_top = scorer.tokenizer.decode([int(logits.argmax())])
    print(f"{it.id}: p_yes={res.p_yes_3way:.3f} mass={res.mass_covered:.3f} "
          f"top={res.top_token!r} manual={manual_top!r}")
    assert manual_top == res.top_token, "final-position read mismatch under device_map"
    assert res.mass_covered > 0.2, "coverage collapsed - thinking mode leaking?"

del scorer, logits, enc
gc.collect()
for i in range(torch.cuda.device_count()):
    torch.cuda.empty_cache()
print("\nOK: final-position logits verified under device_map=auto on 2 items")


In [ ]:
# --- 4. Per model: gate -> score -> coverage check -> baseline -> controls --
import json, pathlib, subprocess, sys, time

def run(label, mod_args):
    print(f"\n=== {label} ===", flush=True)
    t0 = time.time()
    p = subprocess.Popen([sys.executable, "-u", "-m"] + mod_args,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"{label} FAILED (exit {p.returncode})")
    print(f"    [{label}: {time.time()-t0:.0f}s]", flush=True)

def coverage_gate(run_json, floor=0.5):
    meta = json.loads(pathlib.Path(run_json).read_text())["meta"]
    cov, cmin = meta["mass_covered_mean"], meta["mass_covered_min"]
    print(f"coverage after gate: mean={cov:.4f} min={cmin:.4f}")
    if cov < floor:
        raise RuntimeError(
            f"COVERAGE {cov:.4f} < {floor}: the three options are not where "
            "this model's probability mass lives; every renormalized number "
            "from this run would be a ratio of rounding errors. Check "
            "--no-thinking and the option table above.")

ITEMS = ["--items", "items/draft", "items/seed"]
t_start = time.time()

for tag, model in MODELS.items():
    run(f"{tag}: tokenization gate",
        ["src.score", "--model", model, "--third-option", "Unknown",
         "--check-tokenization-only"])
    run(f"{tag}: score 100 items",
        ["src.score", "--model", model, *COMMON, *ITEMS,
         "--checkpoint", f"results/partial_{tag}.jsonl",
         "--min-mass-covered", "0.5",
         "--out", f"results/run_{tag}.json"])
    coverage_gate(f"results/run_{tag}.json")
    run(f"{tag}: baseline (20 claims, no evidence)",
        ["src.baseline", "--model", model, *COMMON, *ITEMS,
         "--checkpoint", f"results/partial_{tag}_baseline.jsonl",
         "--out", f"results/baseline_{tag}.json"])
    run(f"{tag}: control - shuffled case order",
        ["src.score", "--model", model, *COMMON, *ITEMS, "--shuffle-cases",
         "--checkpoint", f"results/partial_{tag}_shuffled.jsonl",
         "--out", f"results/run_{tag}_shuffled.json"])
    run(f"{tag}: control - option rotations (3 passes/item)",
        ["src.score", "--model", model, *COMMON, *ITEMS, "--option-rotations",
         "--checkpoint", f"results/partial_{tag}_rotations.jsonl",
         "--out", f"results/run_{tag}_rotations.json"])
    run(f"{tag}: analysis",
        ["src.analyze", "--run", f"results/run_{tag}.json",
         "--baseline", f"results/baseline_{tag}.json",
         "--out", f"results/analysis_{tag}.json"])

print(f"\nall stages done in {(time.time()-t_start)/60:.0f} min")
for tag in MODELS:
    md = pathlib.Path(f"results/analysis_{tag}.md").read_text()
    head = md.split("## 2.")[0]
    print(f"\n{'='*72}\nHEADLINE - {tag}\n{'='*72}\n{head}")


In [ ]:
# --- 5. Zip results/ for download ------------------------------------------
import pathlib, shutil

zip_path = shutil.make_archive(
    "/kaggle/working/coherence_confidence_results", "zip", "results")
mb = pathlib.Path(zip_path).stat().st_size / 1e6
print(f"wrote {zip_path} ({mb:.1f} MB)")
print("Download: notebook viewer -> Output tab -> coherence_confidence_results.zip")
